## 0. Setup

In [1]:
from pathlib import Path
import os
import random
import logging
import yaml
import numpy as np
import pandas as pd
import torch

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Using PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
from torch.utils.data import DataLoader
from sklearn.model_selection import GroupKFold, cross_val_score, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from transformers import AutoTokenizer, AutoModel

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Robustly find project root
current_path = Path(".").resolve()
if (current_path / "src").exists():
    PROJECT_ROOT = current_path
else:
    PROJECT_ROOT = current_path.parent

print(f"Project root set to: {PROJECT_ROOT}")

Using PyTorch version: 2.5.1+cu121
CUDA available: True
Project root set to: C:\Users\jhon0\Documents\UTP\Work\MultimodalAnalysis


## 1. Questionnaire Preprocessing

In [2]:
cfg_path = PROJECT_ROOT / "configs" / "preprocessing.yaml"
with open(cfg_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

q = cfg["questionnaire"]
survey_path = PROJECT_ROOT / q["raw_data_path"]
survey = pd.read_excel(survey_path, sheet_name="Demographics")

survey = survey.drop(q["rows_to_drop"], axis=0)
survey = survey.fillna(q["fill_value"])
survey["Gender"] = survey["Gender"].replace(q["gender_mapping"])

pain = q["pain_scale"]
survey["Pain Scale"] = pd.cut(
    survey[pain["source_column"]],
    bins=pain["bins"],
    labels=pain["labels"],
).astype("object")

drop_cols = list(q["columns_to_drop"]) + [pain["source_column"]]
survey = survey.drop(drop_cols, axis=1)

survey_numeric = survey.copy()
for col in survey.columns:
    if not pd.api.types.is_numeric_dtype(survey[col]):
        survey_numeric[col] = pd.Categorical(
            survey[col], categories=survey[col].unique()
        ).codes

## 1.1 Survey (RF)

In [3]:
y = survey_numeric["Pain Scale"]
x = survey_numeric.drop(["ID", "Pain Scale"], axis=1)

print(f"Features (x):\n", list(x.columns))
print(f"\nShape x: {x.shape} | Shape y: {y.shape}")
print(f"Classes in target (y): {y.unique()}")

RF_model = RandomForestClassifier(n_estimators=100, random_state=42)
k_folds = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(RF_model, x, y, cv=k_folds)
print(f"\nCross-validation scores: {scores.mean():.4f} ± {scores.std():.4f}")

Features (x):
 ['Age', 'Gender', 'Etiology of NP*', 'Time with NP', 'Medical treatment for NP', 'Since when have you use the previous medication?', 'Have you had any medical procedures to control the pain?', 'Do you go regularly to phsychological or emotional counseling sessions for your pain?']

Shape x: (36, 8) | Shape y: (36,)
Classes in target (y): [0 1 2]

Cross-validation scores: 0.4143 ± 0.1143


### 1.2 Survey (SVM - BERT)

In [4]:
survey_sentence = []
for _, row in survey.iterrows():
    age_val = row["Age"]
    try:
        age = int(age_val)
    except (TypeError, ValueError):
        age = age_val

    sentence = (
        f"The patient with ID {row['ID']} is {age} years old and identifies as {row['Gender']}. "
        f"The etiology of neuropathic pain is {row['Etiology of NP*']}. "
        f"Pain duration: {row['Time with NP']}. "
        f"Current treatment: {row['Medical treatment for NP']}. "
        f"Medication since {row['Since when have you use the previous medication?']}. "
        f"Medical procedures: {row['Have you had any medical procedures to control the pain?']}. "
        f"Psychological or emotional counseling: {row['Do you go regularly to phsychological or emotional counseling sessions for your pain?']}."
    )
    survey_sentence.append(sentence)

survey_text = []
for patient_id in range(survey.shape[0]):
    text = ""
    for column in survey.columns:
        text = text + str(survey.iloc[patient_id][column]) + "\n"
    survey_text.append(text)

texts_for_embedding = survey_text
texts_for_embedding[:2]

['0\n25.0\nFemenine\nCentral Nervous System Disorder (CRPS or Lyme)\nMore than 2 years\nPregabalin, amitriptyline\nMore than a year ago\nNerve blocks and infusions*\nYes\nSevere\n',
 '1\n57.0\nFemenine\nDiabetes\nMore than 2 years\nTramadol\nMore than a month ago\nUnknown\nNo\nModerate\n']

In [5]:
from transformers import logging as hf_logging

hf_logging.set_verbosity_error()

model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.add_special_tokens({"pad_token": "[PAD]"})
bert_model = AutoModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model.to(device)
bert_model.eval();

2026-05-12 15:41:59,137 - INFO - HTTP Request: HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/config.json "HTTP/1.1 200 OK"
2026-05-12 15:41:59,139 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-12 15:41:59,286 - INFO - HTTP Request: HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-12 15:41:59,391 - INFO - HTTP Request: GET https://huggingface.co/api/models/bert-base-multilingual-cased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-05-12 15:41:59,581 - INFO - HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-multilingual-cased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-12 15:41:59,682 - INFO - HTTP Request: GET https://huggingface.co/api/models/bert-bas

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [6]:
def encode_texts(texts, batch_size=1, max_length=512):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = bert_model(**inputs)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]
        embeddings.append(cls_embeddings.cpu().numpy())
    return np.vstack(embeddings)

survey_embedding = encode_texts(texts_for_embedding)

y = survey["Pain Scale"]
x = survey_embedding

svm_model = SVC()
kfold = KFold(n_splits=5, shuffle=False)
scores = cross_val_score(svm_model, x, y, cv=kfold)
print(f"Cross Validation Score: {scores.mean():.4f} ± {scores.std():.4f}")

Cross Validation Score: 0.6071 ± 0.1464


## 2. EEG preprocessing

In [7]:
from src.config import EEGPreprocessingConfig
from src.preprocessing import EEGProcessor
import mne

cfg = EEGPreprocessingConfig.from_yaml(
    PROJECT_ROOT / "configs" / "preprocessing.yaml",
    base_dir=PROJECT_ROOT,
)
processor = EEGProcessor(cfg)

gdf_files = sorted(cfg.raw_data_dir.glob("*.gdf"))
print(f"GDF files: {len(gdf_files)}")

if gdf_files:
    raw = mne.io.read_raw_gdf(gdf_files[0], preload=False, verbose=False)
    print("Sample file:", gdf_files[0].name)
    print("Channel count:", len(raw.ch_names))
    print("Channel types:", sorted(set(raw.get_channel_types())))
    print("First 10 channels:", raw.ch_names[:10])
    print("sfreq:", raw.info["sfreq"])
    print("duration_sec:", raw.times[-1])

X_all, y_open_closed, ids_all = processor.build_open_closed_dataset()
if X_all.size == 0:
    raise ValueError("No EEG epochs found.")

pain_cats = ["Low", "Moderate", "Severe"]
pain_codes = pd.Categorical(survey["Pain Scale"], categories=pain_cats).codes
id_to_pain = dict(zip(survey["ID"].astype(int), pain_codes))

valid_mask = np.array(
    [int(pid) in id_to_pain and id_to_pain[int(pid)] >= 0 for pid in ids_all]
)
X_all = X_all[valid_mask]
y_open_closed = y_open_closed[valid_mask]
ids_all = ids_all[valid_mask]
y_pain_all = np.array([id_to_pain[int(pid)] for pid in ids_all], dtype=int)

open_mask = y_open_closed == 0
X_open = X_all[open_mask]
y_open = y_pain_all[open_mask]
ids_open = ids_all[open_mask]

closed_mask = y_open_closed == 1
X_closed = X_all[closed_mask]
y_closed = y_pain_all[closed_mask]
ids_closed = ids_all[closed_mask]

print("Epochs (total):", X_all.shape)
print("Pain scale counts:", np.bincount(y_pain_all))
print("Open epochs:", X_open.shape, "Closed epochs:", X_closed.shape)


GDF files: 37
Sample file: ID0.gdf
Channel count: 24
Channel types: ['eeg']
First 10 channels: ['FP1', 'FP2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2']
sfreq: 250.0
duration_sec: 599.804
Epochs (total): (22124, 13, 256)
Pain scale counts: [6577 6578 8969]
Open epochs: (11063, 13, 256) Closed epochs: (11061, 13, 256)


In [8]:
from src.training import train_eegformer, extract_embeddings

model_open = train_eegformer(X_open, y_open, seed=SEED)
emb_open = extract_embeddings(model_open, X_open)

model_closed = train_eegformer(X_closed, y_closed, seed=SEED)
emb_closed = extract_embeddings(model_closed, X_closed)

model_all = train_eegformer(X_all, y_pain_all, seed=SEED)
emb_all = extract_embeddings(model_all, X_all)

### 2.1 EEG open (EEGFormer - SVM)

In [11]:
cv_open = GroupKFold(n_splits=min(5, len(np.unique(ids_open))))
scores = cross_val_score(
    SVC(kernel="rbf"), emb_open, y_open, cv=cv_open, groups=ids_open
)
print(f"Cross Validation Score: {scores.mean():.4f} ± {scores.std():.4f}")

Cross Validation Score: 0.4556 ± 0.1612


### 2.2 EEG closed (EEGFormer - SVM)

In [12]:
cv_closed = GroupKFold(n_splits=min(5, len(np.unique(ids_closed))))
scores = cross_val_score(
    SVC(kernel="rbf"), emb_closed, y_closed, cv=cv_closed, groups=ids_closed
)
print(f"Cross Validation Score: {scores.mean():.4f} ± {scores.std():.4f}")


Cross Validation Score: 0.4254 ± 0.0947


### 2.3 EEG complete (EEGFormer - SVM)

In [13]:
cv_all = GroupKFold(n_splits=min(5, len(np.unique(ids_all))))
scores = cross_val_score(
    SVC(kernel="rbf"), emb_all, y_pain_all, cv=cv_all, groups=ids_all
)
print(f"Cross Validation Score: {scores.mean():.4f} ± {scores.std():.4f}")

Cross Validation Score: 0.4228 ± 0.0895


## 3. Concatenate embeddings

### 3.1 Questionnaires + EEG open (BERT - EEGFormer - SVM)

In [14]:
id_to_bert = dict(zip(survey["ID"].astype(int), survey_embedding))
open_mask = np.array([int(pid) in id_to_bert for pid in ids_open])
emb_open_f = emb_open[open_mask]
y_open_f = y_open[open_mask]
ids_open_f = ids_open[open_mask]

if len(ids_open_f) == 0:
    raise ValueError("No matching questionnaire embeddings for EEG open.")

bert_open = np.stack([id_to_bert[int(pid)] for pid in ids_open_f])
x_open_concat = np.concatenate([emb_open_f, bert_open], axis=1)

cv_open_concat = GroupKFold(n_splits=min(5, len(np.unique(ids_open_f))))
scores = cross_val_score(
    SVC(kernel="rbf"), x_open_concat, y_open_f, cv=cv_open_concat, groups=ids_open_f
)
print(
    f"Cross Validation Score (EEG open + BERT): {scores.mean():.4f} ± {scores.std():.4f}"
)


Cross Validation Score (EEG open + BERT): 0.9570 ± 0.0498


### 3.1 Questionnaires + EEG closed (BERT - EEGFormer - SVM)

In [15]:
id_to_bert = dict(zip(survey["ID"].astype(int), survey_embedding))
closed_mask = np.array([int(pid) in id_to_bert for pid in ids_closed])
emb_closed_f = emb_closed[closed_mask]
y_closed_f = y_closed[closed_mask]
ids_closed_f = ids_closed[closed_mask]

if len(ids_closed_f) == 0:
    raise ValueError("No matching questionnaire embeddings for EEG closed.")

bert_closed = np.stack([id_to_bert[int(pid)] for pid in ids_closed_f])
x_closed_concat = np.concatenate([emb_closed_f, bert_closed], axis=1)

cv_closed_concat = GroupKFold(n_splits=min(5, len(np.unique(ids_closed_f))))
scores = cross_val_score(
    SVC(kernel="rbf"), x_closed_concat, y_closed_f, cv=cv_closed_concat, groups=ids_closed_f
)
print(
    f"Cross Validation Score (EEG closed + BERT): {scores.mean():.4f} ± {scores.std():.4f}"
)


Cross Validation Score (EEG closed + BERT): 0.9332 ± 0.1106


### 3.1 Questionnaires + EEG complete (BERT - EEGFormer - SVM)

In [16]:
id_to_bert = dict(zip(survey["ID"].astype(int), survey_embedding))
all_mask = np.array([int(pid) in id_to_bert for pid in ids_all])
emb_all_f = emb_all[all_mask]
y_all_f = y_pain_all[all_mask]
ids_all_f = ids_all[all_mask]

if len(ids_all_f) == 0:
    raise ValueError("No matching questionnaire embeddings for EEG complete.")

bert_all = np.stack([id_to_bert[int(pid)] for pid in ids_all_f])
x_all_concat = np.concatenate([emb_all_f, bert_all], axis=1)

cv_all_concat = GroupKFold(n_splits=min(5, len(np.unique(ids_all_f))))
scores = cross_val_score(
    SVC(kernel="rbf"), x_all_concat, y_all_f, cv=cv_all_concat, groups=ids_all_f
)
print(
    f"Cross Validation Score (EEG complete + BERT): {scores.mean():.4f} ± {scores.std():.4f}"
)


Cross Validation Score (EEG complete + BERT): 0.9170 ± 0.1119


## 4. Latent space

In [ ]:
from src.utils.visualization import plot_latent_space_3d
import plotly.io as pio

label_names = {0: "Low", 1: "Moderate", 2: "Severe"}

id_to_bert = dict(zip(survey["ID"].astype(int), survey_embedding))

pio.renderers.default = "vscode"


def build_fused(emb_eeg, y_eeg, ids_eeg, label):
    mask = np.array([int(pid) in id_to_bert for pid in ids_eeg])
    emb_f = emb_eeg[mask]
    y_f = y_eeg[mask]
    ids_f = ids_eeg[mask]
    if len(ids_f) == 0:
        raise ValueError(f"No matching questionnaire embeddings for {label}.")
    bert = np.stack([id_to_bert[int(pid)] for pid in ids_f])
    return np.concatenate([emb_f, bert], axis=1), y_f

x_open_concat, y_open_f = build_fused(emb_open, y_open, ids_open, "EEG open")
x_closed_concat, y_closed_f = build_fused(emb_closed, y_closed, ids_closed, "EEG closed")
x_all_concat, y_all_f = build_fused(emb_all, y_pain_all, ids_all, "EEG complete")

lat_q = survey_embedding
lat_eeg_open = emb_open
lat_eeg_closed = emb_closed
lat_fused_open = x_open_concat
lat_fused_closed = x_closed_concat
lat_fused_completed = x_all_concat

print("Generating Latent Space Visualizations...")
cases_visual = [
    (lat_q, pain_codes, "Latent Space: Questionnaires (Clinical BERT)"),
    (lat_eeg_open, y_open, "Latent Space: EEG Open Eyes"),
    (lat_eeg_closed, y_closed, "Latent Space: EEG Closed Eyes"),
    (lat_fused_open, y_open_f, "Latent Space: Fused (Q + Open)"),
    (lat_fused_closed, y_closed_f, "Latent Space: Fused (Q + Closed)"),
    (lat_fused_completed, y_all_f, "Latent Space: Fused (Complete)"),
]

for data, labels, title in cases_visual:
    named_labels = [label_names.get(int(l), str(l)) for l in labels]
    fig = plot_latent_space_3d(data, named_labels, title)
    fig.show()

Generating Latent Space Visualizations...


In [ ]:
import re
# Save visualizations to HTML files
output_dir = PROJECT_ROOT / "outputs" / "visualizations"
output_dir.mkdir(parents=True, exist_ok=True)

for i, (data, labels, title) in enumerate(cases_visual, start=1):
    named_labels = [label_names.get(int(l), str(l)) for l in labels]
    fig = plot_latent_space_3d(data, named_labels, title)

    filename = re.sub(r"[^a-zA-Z0-9]+", "_", title.lower()).strip("_")
    html_path = output_dir / f"{i:02d}_{filename}.html"

    fig.write_html(str(html_path), include_plotlyjs="cdn", full_html=True)
